## Setup

In [ ]:
# Import Libraries 

import os                                       # Operating System library (here used to find number of files)
import numpy as np                              # Numpy library for numerical operations                       
import matplotlib.pyplot as plt                 # Matplotlib library for plotting        
import scipy as sp                              # Scipy library for scientific operations                                    

## Accessing Data Locally or with Google Colab 

In [ ]:
# Get directory on local machine 

data_directory = os.path.join(os.getcwd(), "pump_data")                         # Directory where data is stored

In [ ]:
# Get directory in Google Docs - NOTE: Create Dataset code below will need to be modified to work with Google Drive 

from google.colab import drive                                                  # Import Google Drive library

drive.mount('/content/drive')                                                   # Mount Google Drive
data_directory = "/content/drive/My Drive/case1/pump_data"                      # TODO: Change the path according to your google drive

## Create Dataset

In [ ]:
# function to import data from one file 

def get_pump_data(file_number):
    file_name = data_directory + "/Data - pump " + str(file_number) + ".csv"    # create filename from file number 
    pump_data = np.genfromtxt(file_name, delimiter=",")                         # import data from file
    pump_data = np.delete(pump_data, 0, axis=0)                                 # remove first row (headers)

    return pump_data


In [ ]:
# find out how many CSV files we have 

files = os.listdir(data_directory)
csv_files = [f for f in files if (f.find(".csv") > 0)]
num_csv_files = len(csv_files)

print(num_csv_files)

In [ ]:
# load data from all files 

pumps = []

for n in range(num_csv_files):
    pump = get_pump_data(n+1)
    pumps.append(pump)


## Calculate Residuals

In [ ]:
# define "ideal" model / functions 

def pump_model_QH(Q,rpm):
    QH_coeff = [-0.580818, 1.65095, -1.4245, 0.105362, 0.104778, 0.249879]
    nref = .5
    QH_func = np.poly1d(QH_coeff)
    H = QH_func(Q)*(rpm/nref)**2
    return H

In [ ]:
# Calculate residuals for a given pump

n = 4                                               # pump number

Q = pumps[n][:, 0]                                  # get measured values of Q, H and rpm for selected pump
H = pumps[n][:, 1]
rpm = pumps[n][:, 3]

H1 = pump_model_QH(Q, rpm)                          # create ideal H values given our dataset of Q and rpm values 

HR = H - H1                                         # calculate residuals

print(HR.size)

In [ ]:
# Visualize Residuals 

plt.scatter(Q, HR, c=rpm)
plt.show() 

## SciPy Curve Fitting 

In [ ]:
from scipy.optimize import curve_fit                            # Import the curve fitting function from scipy

# Define a function to define the desired equation, in this case a 3 order polynomial
def func(xy, a, b, c, d, e, f, g, h, i, j):                     # Define the function with the input xy and the coefficients a, b, c, d, e, f, g, h, i, j
    x, y = xy                                                   # Unpack the x and y values from the input
    return a + b*x + c*y + d*x**2 + e*y**2 + f*x*y + g*x**3 + h*y**3 + i*x*y**2 + j*y*x**2

In [ ]:
# SciPy Curve Fitting

coeff, pcov = curve_fit(func, (Q, rpm), HR)                     # Get the coefficients by using scipy's curve_fit function
print(coeff)

## Visualize data 

In [ ]:
# Plot residual data and the fitted curve

Hfit = func((Q, rpm), *coeff)                   # Calculate the fitted H values using the obtained coefficients   

plt.scatter(Q, HR, c='r')                       # plot residual data in red
plt.scatter(Q, Hfit, c='b')                     # plot fitted data in blue 

plt.show()    

## Rough calculation of variability 

In [ ]:
HT = H1 + Hfit                              # H predicted value is ideal value + residuals

# Plot measured data and the fitted curve

plt.scatter(Q, H, c='r')                    # plot measured data in red
plt.scatter(Q, HT, c='b')                   # plot fitted data in blue

plt.show()    

In [ ]:
var = ((np.abs(HT-H))/H).max()                                  # Calculate the maximum variation between the measured and predicted values in %
var *=100
print(f'pump {n:2}: max variation = {var:.4f} %')

In [ ]:
abs_var = (np.abs(HT-H)).max()                                  # Calculate the maximum variation between the measured and predicted values
print(f'pump {n:2}: max absolute variation = {abs_var:.4f}')